In [1]:
import sys
print(sys.executable)


/home/gpu4080/research/cropscience/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import gc

# =======================
# PATHS (서버)
# =======================
RICE_ROOT = Path("/home/gpu4080/ygdata/rice")

BEST_PATH      = RICE_ROOT / "best_all_sites_DOY.csv"
UNION_GDD_PATH = RICE_ROOT / "1997_2024_RICE_union_all_sites_with_GDD10_since_gs.csv"
PEST_GLOB      = str(RICE_ROOT / "joined_SAMPLE_GDD_1997_2024_RICE_*_plus_ASOS3NN.csv")

OUT_LONG_PATH  = RICE_ROOT / "RICE_LONG_DOY_with_best_and_GDD10_since_gs.csv"

# =======================
# SETTINGS
# =======================
CHUNK_PEST  = 300_000
CHUNK_UNION = 500_000

# 이화명나방은 1/2화기 두 개로 LONG에 풀기
PEST_COL_RULES = {
    "잎집무늬마름병": ["잎집무늬마름병-발생면적률(%)"],
    "흰잎마름병":     ["흰잎마름병-발생면적률(%)"],
    "깨씨무늬병":     ["깨씨무늬병-발생면적률(%)"],
    "벼멸구":         ["벼멸구-발생면적률(%)"],
    "흰등멸구":       ["흰등멸구-발생면적률(%)"],
    "잎도열병":       ["잎도열병-발생면적률(%)"],
    "이화명나방":     ["이화명나방1화기-발생면적률(%)", "이화명나방2화기-발생면적률(%)"],
}

# =======================
# UTILS
# =======================
def read_csv_flexible(p: Path, **kwargs) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            return pd.read_csv(p, encoding=enc, low_memory=False, **kwargs)
        except Exception:
            continue
    return pd.read_csv(p, low_memory=False, **kwargs)

def to_day(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.normalize()

def norm_siteid(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    x = x.str.replace(r"\.0$", "", regex=True)
    return x

def pick_pest_from_filename(fname: str):
    for k in ["잎집무늬마름병","흰잎마름병","깨씨무늬병","벼멸구","흰등멸구","잎도열병","이화명나방"]:
        if k in fname:
            return k
    return None

# =======================
# 0) sanity check
# =======================
assert BEST_PATH.exists(), f"BEST not found: {BEST_PATH}"
assert UNION_GDD_PATH.exists(), f"UNION_GDD not found: {UNION_GDD_PATH}"

pest_files = sorted(glob.glob(PEST_GLOB))
assert pest_files, f"PEST files not found: {PEST_GLOB}"
print("[INFO] pest files:", len(pest_files))

# =======================
# 1) LOAD BEST (DOY 기반)
# =======================
best = read_csv_flexible(BEST_PATH, dtype={"site_id": "string"})

need_best_cols = [
    "site_id","year","좌표-위도","좌표-경도",
    "best_suitability","best_months","offset_days","window_idx",
    "growing_start_doy","growing_end_doy","growing_mid_doy","growing_len_days"
]
missing = [c for c in need_best_cols if c not in best.columns]
if missing:
    raise ValueError(f"[ERROR] BEST 파일에 필요한 컬럼이 없음: {missing}\nBEST columns={list(best.columns)}")

best["site_id"] = norm_siteid(best["site_id"])
best["year"] = pd.to_numeric(best["year"], errors="coerce").astype("Int64")
best = best.dropna(subset=["site_id","year"]).copy()
best["year"] = best["year"].astype(int)

best = best[need_best_cols].copy()
num_cols = ["best_suitability","best_months","offset_days","window_idx",
            "growing_start_doy","growing_end_doy","growing_mid_doy","growing_len_days",
            "좌표-위도","좌표-경도"]
for c in num_cols:
    best[c] = pd.to_numeric(best[c], errors="coerce")

# (중요) site_id-year 중복이 있으면 merge 시 행이 폭증할 수 있으니 방지
best = best.drop_duplicates(["site_id","year"], keep="last").copy()

print("[INFO] best rows:", len(best),
      "sites:", best["site_id"].nunique(),
      "years:", best["year"].nunique())

# =======================
# 2) PEST OBS 추출 (관측 있는 행만 -> LONG base)
# =======================
obs_parts = []
obs_rows = 0

for fp in pest_files:
    fname = Path(fp).name
    pest_key = pick_pest_from_filename(fname)
    if pest_key is None:
        print("[SKIP] cannot infer pest from filename:", fname)
        continue

    # 헤더 체크
    head = read_csv_flexible(Path(fp), nrows=5)
    if "지점ID" not in head.columns or "일시" not in head.columns:
        raise ValueError(f"[ERROR] {fname}: expected columns '지점ID','일시' not found. cols={list(head.columns)}")

    obs_cols = PEST_COL_RULES[pest_key]
    for c in obs_cols:
        if c not in head.columns:
            raise ValueError(f"[ERROR] {fname}: obs col missing: {c}\ncols={list(head.columns)}")

    usecols = ["지점ID", "일시"] + obs_cols
    print(f"[INFO] scan {fname}  pest={pest_key}  obs_cols={obs_cols}")

    for ch in read_csv_flexible(Path(fp), usecols=usecols, chunksize=CHUNK_PEST):
        ch["지점ID"] = norm_siteid(ch["지점ID"])
        ch["_date"] = to_day(ch["일시"])
        ch = ch[ch["_date"].notna()].copy()
        if ch.empty:
            del ch
            gc.collect()
            continue

        if pest_key == "이화명나방":
            for col in obs_cols:
                sub = ch[["지점ID","_date", col]].copy()
                sub[col] = pd.to_numeric(sub[col], errors="coerce")
                sub = sub[sub[col].notna()].copy()
                if sub.empty:
                    continue
                pest_name = "이화명나방1화기" if "1화기" in col else "이화명나방2화기"
                sub = sub.rename(columns={"지점ID":"site_id", col:"obs_value"})
                sub["pest"] = pest_name
                sub["year"] = sub["_date"].dt.year.astype(int)
                sub["obs_doy"] = sub["_date"].dt.dayofyear.astype(int)

                obs_parts.append(sub[["site_id","_date","year","obs_doy","pest","obs_value"]])
                obs_rows += len(sub)
        else:
            col = obs_cols[0]
            sub = ch[["지점ID","_date", col]].copy()
            sub[col] = pd.to_numeric(sub[col], errors="coerce")
            sub = sub[sub[col].notna()].copy()
            if sub.empty:
                del ch
                gc.collect()
                continue

            sub = sub.rename(columns={"지점ID":"site_id", col:"obs_value"})
            sub["pest"] = pest_key
            sub["year"] = sub["_date"].dt.year.astype(int)
            sub["obs_doy"] = sub["_date"].dt.dayofyear.astype(int)

            obs_parts.append(sub[["site_id","_date","year","obs_doy","pest","obs_value"]])
            obs_rows += len(sub)

        del ch
        gc.collect()

print("[INFO] extracted obs rows (pre-concat):", f"{obs_rows:,}")

obs = pd.concat(obs_parts, ignore_index=True) if obs_parts else pd.DataFrame()
del obs_parts
gc.collect()

if obs.empty:
    raise SystemExit("[STOP] obs rows=0. (관측 있는 행이 하나도 안 뽑혔음)")

print("[INFO] obs rows:", len(obs),
      "sites:", obs["site_id"].nunique(),
      "pests:", obs["pest"].nunique())

# =======================
# 3) best merge (관측이 더 중요) -> LEFT JOIN
# =======================
df = obs.merge(best, on=["site_id","year"], how="left")
print("[INFO] after merge(best,left):", len(df),
      "missing best rows:", int(df["best_suitability"].isna().sum()))
del obs
gc.collect()

# =======================
# 4) UNION에서 필요한 (site,date)만 스캔해서 GDD10_since_gs 붙이기 (merge 방식)
# =======================
u_head = read_csv_flexible(UNION_GDD_PATH, nrows=5)
need_cols = {"지점ID","일시","GDD10_since_gs"}
if not need_cols.issubset(set(u_head.columns)):
    raise ValueError(f"[ERROR] UNION columns mismatch. need {sorted(list(need_cols))}\ncols={list(u_head.columns)}")

keys_df = df[["site_id", "_date"]].drop_duplicates().copy()

gdd_hits = []
seen = 0
hit = 0

for i, ch in enumerate(
    read_csv_flexible(
        UNION_GDD_PATH,
        usecols=["지점ID","일시","GDD10_since_gs"],
        chunksize=CHUNK_UNION
    ),
    start=1
):
    ch["지점ID"] = norm_siteid(ch["지점ID"])
    ch["_date"] = to_day(ch["일시"])
    ch = ch[ch["_date"].notna()].copy()
    if ch.empty:
        del ch
        gc.collect()
        continue

    sub = ch.merge(
        keys_df,
        left_on=["지점ID", "_date"],
        right_on=["site_id", "_date"],
        how="inner"
    )

    if not sub.empty:
        sub["GDD10_since_gs"] = pd.to_numeric(sub["GDD10_since_gs"], errors="coerce")
        gdd_hits.append(sub[["site_id","_date","GDD10_since_gs"]])
        hit += len(sub)

    seen += len(ch)
    if i % 5 == 0:
        print(f"[PROGRESS] union chunks={i} scanned_rows={seen:,} hits={hit:,}")

    del ch
    gc.collect()

if gdd_hits:
    gdd_df = pd.concat(gdd_hits, ignore_index=True)
    gdd_df = gdd_df.drop_duplicates(["site_id","_date"], keep="last")
else:
    gdd_df = pd.DataFrame(columns=["site_id","_date","GDD10_since_gs"])

df = df.merge(gdd_df, on=["site_id","_date"], how="left")
df["GDD10_since_gs"] = pd.to_numeric(df["GDD10_since_gs"], errors="coerce").astype("float32").round(1)

print("[INFO] after merge(GDD): missing GDD rows:", int(df["GDD10_since_gs"].isna().sum()))
del gdd_hits, gdd_df, keys_df
gc.collect()

# =======================
# 5) DOY 기반 파생 + label
# =======================
df["label_event"] = (pd.to_numeric(df["obs_value"], errors="coerce").fillna(0) > 0).astype("int8")

for c in ["growing_start_doy","growing_end_doy","growing_mid_doy","growing_len_days"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["days_since_growing_start"] = (df["obs_doy"] - df["growing_start_doy"]).astype("float32")
df["days_until_growing_end"]   = (df["growing_end_doy"] - df["obs_doy"]).astype("float32")

# best가 없는 행은 is_growing을 NA로 두는 게 자연스러움
valid_ge = df["growing_start_doy"].notna() & df["growing_end_doy"].notna()
cond_in  = valid_ge & (df["obs_doy"] >= df["growing_start_doy"]) & (df["obs_doy"] <= df["growing_end_doy"])
df["is_growing"] = np.where(valid_ge, np.where(cond_in, 1, 0), np.nan).astype("float32")

# =======================
# 6) FINAL (date 제거, DOY로만 저장)
# =======================
df = df.drop(columns=["_date"], errors="ignore")

front = ["site_id","좌표-위도","좌표-경도","year","pest","obs_doy","obs_value","label_event","GDD10_since_gs"]
best_block = ["best_suitability","best_months","offset_days","window_idx",
              "growing_start_doy","growing_end_doy","growing_mid_doy","growing_len_days",
              "days_since_growing_start","days_until_growing_end","is_growing"]

front = [c for c in front if c in df.columns]
best_block = [c for c in best_block if c in df.columns]
rest = [c for c in df.columns if c not in set(front + best_block)]

df = df[front + best_block + rest].copy()

df.to_csv(OUT_LONG_PATH, index=False, encoding="utf-8-sig")
print("[SAVED]", OUT_LONG_PATH)
print("rows:", len(df), "cols:", df.shape[1])
print(df.head())

[INFO] pest files: 7
[INFO] best rows: 72661 sites: 2771 years: 28
[INFO] scan joined_SAMPLE_GDD_1997_2024_RICE_깨씨무늬병_plus_ASOS3NN.csv  pest=깨씨무늬병  obs_cols=['깨씨무늬병-발생면적률(%)']
[INFO] scan joined_SAMPLE_GDD_1997_2024_RICE_벼멸구_plus_ASOS3NN.csv  pest=벼멸구  obs_cols=['벼멸구-발생면적률(%)']
[INFO] scan joined_SAMPLE_GDD_1997_2024_RICE_이화명나방_plus_ASOS3NN.csv  pest=이화명나방  obs_cols=['이화명나방1화기-발생면적률(%)', '이화명나방2화기-발생면적률(%)']
[INFO] scan joined_SAMPLE_GDD_1997_2024_RICE_잎도열병_plus_ASOS3NN.csv  pest=잎도열병  obs_cols=['잎도열병-발생면적률(%)']
[INFO] scan joined_SAMPLE_GDD_1997_2024_RICE_잎집무늬마름병_plus_ASOS3NN.csv  pest=잎집무늬마름병  obs_cols=['잎집무늬마름병-발생면적률(%)']
[INFO] scan joined_SAMPLE_GDD_1997_2024_RICE_흰등멸구_plus_ASOS3NN.csv  pest=흰등멸구  obs_cols=['흰등멸구-발생면적률(%)']
[INFO] scan joined_SAMPLE_GDD_1997_2024_RICE_흰잎마름병_plus_ASOS3NN.csv  pest=흰잎마름병  obs_cols=['흰잎마름병-발생면적률(%)']
[INFO] extracted obs rows (pre-concat): 463,889
[INFO] obs rows: 463889 sites: 2772 pests: 8
[INFO] after merge(best,left): 463889 missing best rows: 10

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob

R = Path("/home/gpu4080/ygdata/rice")
OUT = R / "RICE_LONG_DOY_with_best_and_GDD10_since_gs.csv"
BEST = R / "best_all_sites_DOY.csv"
UNION = R / "1997_2024_RICE_union_all_sites_with_GDD10_since_gs.csv"

PEST_COL_RULES = {
    "잎집무늬마름병": ["잎집무늬마름병-발생면적률(%)"],
    "흰잎마름병":     ["흰잎마름병-발생면적률(%)"],
    "깨씨무늬병":     ["깨씨무늬병-발생면적률(%)"],
    "벼멸구":         ["벼멸구-발생면적률(%)"],
    "흰등멸구":       ["흰등멸구-발생면적률(%)"],
    "잎도열병":       ["잎도열병-발생면적률(%)"],
    "이화명나방":     ["이화명나방1화기-발생면적률(%)", "이화명나방2화기-발생면적률(%)"],
}

def read_csv_flexible(p: Path, **kwargs) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            return pd.read_csv(p, encoding=enc, low_memory=False, **kwargs)
        except Exception:
            continue
    return pd.read_csv(p, low_memory=False, **kwargs)

def norm_siteid(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    x = x.str.replace(r"\.0$", "", regex=True)
    return x

def to_day(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.normalize()

def pest_key_from_output(pest: str) -> str:
    return "이화명나방" if pest.startswith("이화명나방") else pest

def obs_col_from_output(pest: str) -> str:
    if pest == "이화명나방1화기":
        return "이화명나방1화기-발생면적률(%)"
    if pest == "이화명나방2화기":
        return "이화명나방2화기-발생면적률(%)"
    return PEST_COL_RULES[pest][0]

def pest_file_for(pest: str) -> Path:
    key = pest_key_from_output(pest)
    cand = sorted(glob.glob(str(R / f"joined_SAMPLE_GDD_1997_2024_RICE_{key}_plus_ASOS3NN.csv")))
    if not cand:
        raise FileNotFoundError(f"pest file not found for pest={pest} (key={key})")
    return Path(cand[0])

# 1) missing 행 뽑기 + 날짜 복원
df = read_csv_flexible(OUT)
miss = df[df["GDD10_since_gs"].isna()].copy()
print("missing rows:", len(miss))
print(miss[["site_id","year","pest","obs_doy","obs_value"]].head())

row = miss.iloc[0]
site = str(row["site_id"])
year = int(row["year"])
doy  = int(row["obs_doy"])
date = (pd.Timestamp(year=year, month=1, day=1) + pd.Timedelta(days=doy-1)).strftime("%Y-%m-%d")
pest = str(row["pest"])
print("\nTARGET ->", {"site_id": site, "year": year, "obs_doy": doy, "date": date, "pest": pest})

# 2) pest 원본 파일에서 “그 행” 찾기
fp = pest_file_for(pest)
obs_col = obs_col_from_output(pest)
print("\n[SEARCH PEST FILE]", fp.name, "obs_col=", obs_col)

found = False
for ch in read_csv_flexible(fp, usecols=["지점ID","일시",obs_col], chunksize=500_000):
    ch["지점ID"] = norm_siteid(ch["지점ID"])
    ch["_date"] = to_day(ch["일시"])
    m = (ch["지점ID"].astype(str) == site) & (ch["_date"] == pd.Timestamp(date))
    if m.any():
        print("\n[FOUND IN PEST FILE]")
        print(ch.loc[m, ["지점ID","일시",obs_col]].head(20))
        found = True
        break
print("found_in_pest_file?", found)

# 3) best에 site-year 있는지 확인
print("\n[CHECK BEST]")
best = read_csv_flexible(BEST, usecols=["site_id","year"])
best["site_id"] = norm_siteid(best["site_id"])
best["year"] = pd.to_numeric(best["year"], errors="coerce")
print("exists_in_best(site-year)?", bool(((best["site_id"].astype(str)==site) & (best["year"]==year)).any()))

# 4) union에 site-date 있는지 확인 (빠르게: 매칭되는지 한 번만 찾고 종료)
print("\n[CHECK UNION]")
hit = False
for ch in read_csv_flexible(UNION, usecols=["지점ID","일시","GDD10_since_gs"], chunksize=700_000):
    ch["지점ID"] = norm_siteid(ch["지점ID"])
    ch["_date"] = to_day(ch["일시"])
    m = (ch["지점ID"].astype(str) == site) & (ch["_date"] == pd.Timestamp(date))
    if m.any():
        print("[FOUND IN UNION]")
        print(ch.loc[m, ["지점ID","일시","GDD10_since_gs"]].head(10))
        hit = True
        break
print("exists_in_union(site-date)?", hit)


missing rows: 1
            site_id  year  pest  obs_doy  obs_value
229944  29446_49518  2000  잎도열병      202       33.3

TARGET -> {'site_id': '29446_49518', 'year': 2000, 'obs_doy': 202, 'date': '2000-07-20', 'pest': '잎도열병'}

[SEARCH PEST FILE] joined_SAMPLE_GDD_1997_2024_RICE_잎도열병_plus_ASOS3NN.csv obs_col= 잎도열병-발생면적률(%)

[FOUND IN PEST FILE]
               지점ID          일시  잎도열병-발생면적률(%)
842452  29446_49518  2000-07-20           33.3
found_in_pest_file? True

[CHECK BEST]
exists_in_best(site-year)? False

[CHECK UNION]
exists_in_union(site-date)? False


In [1]:
# split_long_by_pest.py
from __future__ import annotations

import re
from pathlib import Path
import pandas as pd

IN_PATH = Path("/home/gpu4080/ygdata/rice/RICE_LONG_DOY_with_best_and_GDD10_since_gs.csv")
OUT_DIR = Path("/home/gpu4080/ygdata/rice/LONG_by_pest")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 1_000_000  # 필요하면 줄이기 (예: 300_000)

def slugify(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^\w가-힣\-]+", "", s)  # 한글 유지
    return s[:80] if len(s) > 80 else s

# 1) 헤더에서 병해충 컬럼 후보 찾기
header = pd.read_csv(IN_PATH, nrows=0)
cols = list(header.columns)

candidate_cols = [
    "label_name", "pest", "pest_name", "disease", "target", "TARGET_PEST",
    "병해충", "병해충명", "병명", "해충명", "작물병해충", "target_name"
]
pest_col = None
for c in candidate_cols:
    if c in cols:
        pest_col = c
        break

# 그래도 없으면: 문자열 컬럼들 중 값 종류가 적당히 있는 컬럼을 자동 탐색(첫 chunk만)
if pest_col is None:
    first = pd.read_csv(IN_PATH, chunksize=200_000)
    df0 = next(first)
    obj_cols = [c for c in df0.columns if df0[c].dtype == "object"]
    # 값 종류가 너무 많거나(거의 unique) 너무 적은(상수) 건 제외
    scored = []
    for c in obj_cols:
        nun = df0[c].dropna().astype(str).nunique()
        if 2 <= nun <= 50:  # 병해충 종류 수로 그럴듯한 범위
            scored.append((nun, c))
    scored.sort()
    if scored:
        pest_col = scored[0][1]  # 종류 수가 가장 적은 후보
    else:
        raise RuntimeError(
            f"병해충 식별 컬럼을 못 찾았어. 헤더 컬럼 확인해줘: {cols[:30]} ..."
        )

print(f"[INFO] Using pest column: {pest_col}")

# 2) chunk 단위로 분할 저장
writers: dict[str, Path] = {}
header_written: set[str] = set()
total_rows = 0

for i, chunk in enumerate(pd.read_csv(IN_PATH, chunksize=CHUNKSIZE, low_memory=False)):
    total_rows += len(chunk)
    if pest_col not in chunk.columns:
        raise RuntimeError(f"chunk에 '{pest_col}' 컬럼이 없어. 파일 구조 확인 필요.")

    # pest 값 정리
    chunk[pest_col] = chunk[pest_col].astype(str).str.strip()
    chunk = chunk[chunk[pest_col].notna() & (chunk[pest_col] != "")]

    # 그룹별로 append 저장
    for pest_value, sub in chunk.groupby(pest_col, sort=False):
        slug = slugify(pest_value)
        out_path = OUT_DIR / f"RICE_LONG_{slug}.csv"
        mode = "w" if slug not in header_written else "a"
        sub.to_csv(out_path, index=False, mode=mode, header=(mode == "w"))
        header_written.add(slug)

    if (i + 1) % 5 == 0:
        print(f"[PROGRESS] chunks={i+1} total_rows={total_rows:,} out_dir={OUT_DIR}")

print(f"[DONE] total_rows={total_rows:,} -> saved to {OUT_DIR}")
print("[DONE] created files:")
for p in sorted(OUT_DIR.glob("RICE_LONG_*.csv")):
    print(" -", p.name)


[INFO] Using pest column: pest
[DONE] total_rows=463,888 -> saved to /home/gpu4080/ygdata/rice/LONG_by_pest
[DONE] created files:
 - RICE_LONG_깨씨무늬병.csv
 - RICE_LONG_벼멸구.csv
 - RICE_LONG_이화명나방1화기.csv
 - RICE_LONG_이화명나방2화기.csv
 - RICE_LONG_잎도열병.csv
 - RICE_LONG_잎집무늬마름병.csv
 - RICE_LONG_흰등멸구.csv
 - RICE_LONG_흰잎마름병.csv
